In [ ]:
%matplotlib inline
from matplotlib.animation import FuncAnimation
import matplotlib.pyplot as plt
import seaborn as sns
from programmable_cubes_UDP import ProgrammableCubes
from programmable_cubes_UDP import programmable_cubes_UDP
import numpy as np
from pygmo import problem
import random
from implementation_heuristic import *
from tqdm import tqdm

In [ ]:
PROBLEM = "ISS"
udp = programmable_cubes_UDP(PROBLEM)
udp.fitness(np.array([-1]))
ti = udp.initial_cube_types
ci = udp.final_cube_positions
ct = udp.target_cube_positions
tt = udp.target_cube_types
types = np.arange(np.max(ti)+1)


In [ ]:
def search_and_evaluate(udp:programmable_cubes_UDP,wrong_ids,coordinate_target):
    """ 
    cubes
    wi - ids of cubes that are wrong
    ct - target coords 
    
    """
    # ti = udp.initial_cube_types
    # types = np.arange(np.max(ti)+1)


    cubes = ProgrammableCubes(udp.final_cube_positions)
    chrom = []
    for i in range(len(wrong_ids)):
        id = wrong_ids[i]
        end = coordinate_target[i]
        if not is_connected(end,cubes.cube_position):
            continue

        tmp_chrom, tmp_path, success = axis_search(cubes,id,end,100)
        if not success:
            tmp_chrom, tmp_path, success = astar_cubes(cubes,id,end,100)
        if success:
            chrom.extend(tmp_chrom)
            cubes.apply_chromosome(np.concatenate([tmp_chrom,[-1]]),True)
        else:
            break
    fitness = udp.fitness(np.concatenate([chrom,np.array([-1])]))[0]
    # udp.plot('ensemble', types)
    # udp.plot('target',types)
    return fitness, chrom

def random_positions(max : int):
    return random.randint(0,max), random.randint(0,max)


def swap(a,i,j):
    tmp = a[i]
    a[i] = a[j]
    a[j] = tmp
    return a

def find_chromosome_heuristic(udp:programmable_cubes_UDP):
    """ 
    Idea:
    mark indices at wrong places and with wrong types only once
    try to move the closest
    extend to further apart
    
    """
    # Initialize cubes
    cubes = ProgrammableCubes(udp.final_cube_positions)
    ti = udp.initial_cube_types
    ci = udp.final_cube_positions
    ct = udp.target_cube_positions
    tt = udp.target_cube_types
    types = np.arange(np.max(ti)+1)

    chrom = []

    achieved_fitness_history = []
    fitness_history = []

    # These are part of the structure but with wrong type => need to be moved to allow others move in
    wti = have_wrong_type(cubes.cube_position,ti,ct,tt)
    # wti - wrong type ids, from cube_position
    # These are cubes away from the structure and the hollow points in the structure
    wpi, epi = get_wrong_cube_ids(cubes.cube_position,ct)
    if len(wti) == 0:
        wti = np.array([], dtype=int)
    if len(epi) == 0:
        epi = np.array([], dtype=int)
    if len(wpi) == 0:
        wpi = np.array([], dtype=int)
    # wpi - wrong place ids, from cube_position
    # epi - empty place ids, from target configuration ct

    wrong_ids_initial = np.concatenate([wti,wpi])
    np.random.shuffle(wrong_ids_initial)
    wrong_ids_target = np.concatenate([wti,epi])
    coordinates_target = np.concatenate([ct[epi],ct[wti]])
    np.random.shuffle(coordinates_target)
    prev_wrong_ids_initial = np.array(wrong_ids_initial)
    prev_coordinates_target = np.array(coordinates_target)




    # compute initial fitness

    prev_fitness, chrom = search_and_evaluate(udp,wrong_ids_initial,coordinates_target)
    fitness = None
    achieved_fitness_history.append(prev_fitness)
    fitness_history.append(prev_fitness)

    iter = 10000
    T = 1
    cooling = 0.9
    distance = 3
    length = np.inf
    pbar = tqdm(range(iter))
    for t in pbar:
        wrong_ids_initial = np.array(prev_wrong_ids_initial)
        coordinates_target = np.array(prev_coordinates_target)
        for d in range(distance):
            odd = random.randint(0,1)==0
            i, j = random_positions(len(wrong_ids_initial)-1)
            if odd:
                wrong_ids_initial = swap(wrong_ids_initial, i, j)
            else:
                coordinates_target = swap(coordinates_target,i, j)
        udp.fitness(np.array([-1]))
        fitness, chrom = search_and_evaluate(udp,wrong_ids_initial[:t],coordinates_target[:t])
        achieved_fitness_history.append(fitness)
        
        if prev_fitness >= fitness:# or random.random() < np.exp((prev_fitness-fitness)/T): # prev_fitness < fitness => exp((prev_fitness - fitness)) < 1
            fitness_history.append(fitness)
            prev_fitness = fitness
            prev_coordinates_target = np.array(coordinates_target)
            prev_wrong_ids_initial = np.array(wrong_ids_initial)
        else:
            fitness_history.append(prev_fitness)
        pbar.set_description(f"Best: {prev_fitness}, Current: {fitness}")
        T *= cooling

    plt.plot(fitness_history,label="Best")
    plt.plot(achieved_fitness_history,label="Current")
    plt.legend()
    plt.show()
    return chrom,fitness_history
        
        

udp.fitness(np.array([-1]))
chrom, fitness_history = find_chromosome_heuristic(udp)


    

In [ ]:
udp.plot("ensemble",[0,1,2])